# UK Exports Predictor

**The question.** Government export Advisors must decide which UK industries to help, in which Markets, and why. They mostly work from what the UK already sells. That hides two kinds of lead: Markets where the UK sells far less of a Product than comparable exporters do, and Markets where a Barrier, such as the lack of a Trade Agreement, holds UK exports down.

**The approach.** A gradient-boosted model learns from global HS6 goods trade, 2012–2024 (about 226 Exporters, 5,200 Products and 142 M Flows), what Flow to expect between any Exporter, Importer and Product, given their fundamentals. Applied to the UK, it estimates **Export Potential** for every Product–Market pair. Export Potential minus Actual Exports is the **Export Gap**, and positive gaps ranked by size are **Opportunities**. Re-predicting with only the trade-policy features changed gives the **Scenario Effect** of a Trade Agreement or Trading Bloc **Scenario**, and pairs that a Scenario would unlock materially are **Blocked Opportunities**. A FastAPI service serves the saved model, and Advisors use it through a Streamlit app.

**What we found** (real data; sections 7 and 10 give the detail):

- **Across all Exporters the model clearly beats a naive market-share baseline** on the untouched 2023 test year: within-Product rank correlation 0.71 against 0.56, and about a sixteenth of its deviance.
- **For UK pairs, scored out-of-fold, it beats the naive baseline only narrowly**: 0.54 against 0.52. Its gain is mostly in the *size* of Export Potential (deviance 658 against 3,653), not in the order of Markets.
- **Large Export Gaps point at growth, but mostly because they sit on large pairs.** Lift is 6.6, but against pairs of similar size it is 0.83.
- **Scenario Effects are several times the changes actually observed.** For the EU–Japan agreement the model associates +45% with the agreement; the observed difference-in-differences is +3% (−6% to +15%).
- **BACI's UK total for 2023 is 16% below HMRC's own figure.** Non-monetary gold and trade HMRC does not attribute to a Product or Market explain about four fifths of the gap; the pairs both sources record agree closely in rank (0.89).

So the tool is useful for *where to look*, as a ranking of leads for a person to check, and is not a forecast of what an agreement or an export push would deliver. The language throughout is associational: "the model associates a Trade Agreement with +X%", never "the agreement causes +X%".

**How to read this notebook.** Each section opens with what it does and why, and closes with a **Conclusion**. All reusable logic lives in the `exportspredictor` package, which the API imports too, so training and serving compute features identically; the notebook only calls it. With the default parameter below, every cell runs in seconds on a tiny fixture world. Real-data results are quoted in the prose from the run of 17 November 2026.

**Course concepts applied, and where the project goes beyond them** (`ML_AI_Concept_Notes.html`):

| Section | Course concept applied | Beyond the notes |
|---|---|---|
| 1. Flows | Exploratory data analysis; data quality audit; descriptive statistics (skew) | Lazy Polars scans over 142 M rows; Enum-coded Parquet |
| 2. Gravity and WDI | Feature engineering; imputing missing values with an explicit rule | Primary-source cross-check of BACI against HMRC OTS |
| 3. Policy features | Feature engineering; data lineage | Pair-year policy features from per-member WTO dates, cross-checked with CEPII Gravity |
| 4. Feature table | Data leakage (preprocessing and target); out-of-fold target encoding; class imbalance | Leave-one-out supply and demand; Horvitz–Thompson weighted zero sampling |
| 5. Training | Ensembles (gradient boosting); train/validation/test splits; hyperparameter search; early stopping; loss vs metric; GPUs and device placement | Tweedie loss; out-of-fold cross-fitting for the UK (ADR 0001); a data iterator for data that does not fit in memory |
| 6. Scenarios | Correlation is not causation; the counterfactual test | Monotone constraints on every policy feature (ADR 0002) |
| 7. Evaluation | Baselines first; regression metrics; worst-group evaluation; drift sensitivity | A ranking metric within Product; Lift with a cluster bootstrap and a size-matched control; difference-in-differences Event Backtests |
| 8. Explaining | Global vs local explanations; SHAP and actionable explanations; feature importance | Exact TreeSHAP from each pair's own fold model, in plain words |
| 9. Artefacts | Reproducible artefacts; registry versions | Versioned artefact folders the API serves unchanged |
| 10. Governance | Model cards; data lineage; data and algorithmic bias; representation audit; EU AI Act tiers and intended purpose; the UK approach; PSI drift | Bias review by reporting quality (income level), not only by group size |

## Parameters

`FIXTURE_MODE` is the notebook's one parameter (the cell is tagged `parameters`, so papermill and similar tools can set it).

- `FIXTURE_MODE = True` (the default) runs every section on a tiny synthetic world in a few seconds. The test suite executes the notebook this way (`tests/test_notebook.py`), so a broken cell fails the build. Cells that need the real downloads, such as the HMRC cross-check and the data-lineage fingerprints, say so and skip.
- `FIXTURE_MODE = False` runs on the real data. `scripts/rebuild.py` sets it through the environment variable `EXPORTSPREDICTOR_FIXTURE_MODE=0` after downloading every source into `EXPORTSPREDICTOR_DATA_DIR`. Section 4 then builds the real feature table, section 5 tunes and trains the real models on the GPU, section 7 trains the test-year and backtest models and evaluates them (about an hour in all on an RTX 3080), section 9 saves the artefacts the API serves, and section 10 writes the model card next to them.

In [ ]:
import os

FIXTURE_MODE = True  # True: the tiny fixture world, in seconds. False: the real data (about an hour on the GPU).

if os.environ.get("EXPORTSPREDICTOR_FIXTURE_MODE") == "0":
    FIXTURE_MODE = False  # scripts/rebuild.py sets this for the real run
FIXTURE_MODE

In [ ]:
import polars as pl

from exportspredictor import artefacts, baci, pipeline, policy
from exportspredictor.fixture import build_world
from exportspredictor.paths import artefacts_dir


## 1. The Flows: BACI HS12

Everything the model knows comes from one dataset, so we start by reading it: what it covers, how skewed it is, and what is provisional. The training data is CEPII's BACI, HS 2012 revision, release V202601 (ADR 0003): every non-zero Flow between about 226 Exporters and Importers for about 5,200 Products, 2012–2024. BACI reconciles each country's export report with its partners' import reports, so a UK Flow is a blend of what the UK and the Market declared, not HMRC's figure. Values are converted from thousand USD to USD.

This section is the course's **exploratory data analysis** and **data-quality audit**, applied at a scale where pandas would not fit: the download is converted once from CSV to Parquet with compact types (Enum codes for countries and Products, so `010121` keeps its leading zero), and Polars scans it lazily, so only the columns and years a query needs are read. The conversion peaks at about 2.5 GB of memory.

In fixture mode the same summaries run on the fixture world's Flows.

In [ ]:
if FIXTURE_MODE:
    flows = build_world().flows.lazy()
else:
    print(baci.prepare())  # idempotent: skips the download and conversion once done
    flows = baci.scan_flows()
flows.head(5).collect()

### Coverage and size per year

Row counts per year, with the number of Products, Exporters and Importers that trade, total world value and the UK's share of it. BACI records only strictly positive Flows, so a missing row means zero *or* unreported: the model treats absent pairs as zeros, which matters for poorly reporting countries.

In [ ]:
flow_summary = baci.summarise_flows(flows)
flow_summary

In [ ]:
flows.filter(pl.col("value_usd") > 0).select(
    flows=pl.len(),
    products=pl.col("hs6").n_unique(),
    exporters=pl.col("exporter").n_unique(),
    importers=pl.col("importer").n_unique(),
).collect()

On the real data (V202601) there are about 9.0 M Flows in 2012, rising to 11.8 M in 2023, about 142 M in total. 5,199 of the 5,202 HS 2012 Products trade in a typical year, and 226 Exporters appear each year (227 across all years). The UK's share of world goods exports in BACI drifts down from about 2.4–2.6% in 2012–2016 to about 1.9% in 2021–2023.

Flow values are extremely skewed: the mean UK Flow (about $1.3 M) is roughly 100 times its median (about $14k), which is why the model uses a Tweedie loss rather than squared error on raw values (course: skewed variables favour the median).

In [ ]:
uk_latest = flows.filter((pl.col("exporter").cast(pl.String) == baci.UK) & (pl.col("value_usd") > 0))
uk_latest.group_by("year").agg(
    mean_usd=pl.col("value_usd").mean(),
    median_usd=pl.col("value_usd").median(),
    markets=pl.col("importer").n_unique(),
    products=pl.col("hs6").n_unique(),
).sort("year").collect()

### The provisional 2024 data

BACI's latest year is provisional: Comtrade is downloaded in January, before every country has reported, and CEPII revises it in the next release. The table below compares each year with the one before. On the real data, 2024 has about 5.5% fewer Flows than 2023 while world value is almost flat, consistent with late reporters still missing small Flows. UK exports fall about 8% in 2024, which may partly reflect the same incompleteness. 2024 is the Baseline year, so the app labels it provisional, and the evaluation (test year 2023) never relies on it.

In [ ]:
flow_summary.select(
    "year",
    flows_change=pl.col("flows").pct_change(),
    world_value_change=pl.col("world_value_usd").pct_change(),
    uk_value_change=pl.col("uk_value_usd").pct_change(),
).tail(3)

**Conclusion.** BACI gives a complete, consistent HS 2012 panel of 142 M non-zero Flows, small enough to scan lazily on one machine. Three properties shape every later choice: values are extremely skewed (hence a Tweedie loss in section 5), absent Flows mean zero *or* unreported (hence sampled zeros in section 4 and the reporting-quality bias review in section 10), and 2024 is provisional (hence a 2023 test year and a stability check in section 7). Section 2 checks BACI's UK figures against HMRC's own.

## 2. Gravity and WDI features: coverage

The gravity model of trade (the oldest empirical regularity in trade economics) says Flows grow with the size of both countries and shrink with the distance between them. These are the non-policy fundamentals the model conditions on. This is feature engineering in the course's sense: domain knowledge turned into inputs, built once in the package (`exportspredictor.gravity`) so training and serving share it.

- **Pair features** from CEPII Gravity (V202211): distance between main cities, contiguity, common official language, colonial tie (ever in a colonial or dependency relationship) and common colonizer. They barely change over time, so each pair takes its latest non-null value.
- **Country-year features**: GDP (current USD) and population for every year 2012–2024.

`gravity.build()` downloads what is missing and writes the derived tables; the data rebuild script runs it. It needs BACI's country code file, so it runs after section 1. The cells below run on real data only (`FIXTURE_MODE = False`).

**Country codes.** BACI uses its own numeric codes. All 227 Exporters and Importers in the 2012–2024 Flows map to an ISO3 code, and every one has Gravity pair features. Three mismatches needed a fix:

- BACI 490 "Other Asia, nes" (ISO3 field `S19`) is Taiwan, mapped to `TWN`.
- Saint Barthélemy (`BLM`) and the French Southern and Antarctic Lands (`ATF`) are missing from Gravity. Their pair features come from a stand-in, Guadeloupe and Réunion. Together they carry about $2.6 bn of Flows over 13 years.

**Missing GDP and population: the rule.** For each country and variable:

1. World Bank WDI, the main source.
2. Otherwise the IMF World Economic Outlook. This covers Taiwan (absent from WDI), South Sudan from 2016, Yemen from 2019 and Eritrea to 2019.
3. Otherwise CEPII Gravity's own figures, available up to 2021.
4. Otherwise linear interpolation between the nearest observed years, or the nearest year carried forward or back at the edges. This applies, for example, to Cuba's GDP from 2021 and Syria's from 2023.
5. Otherwise missing. 21 small territories have no GDP in any source, including Anguilla, the Falklands, Gibraltar, North Korea and Saint Helena. Together they account for about 0.1% of world Flow value. XGBoost learns a default direction for missing values at each split, so these rows still train. Imputing a guess would add noise without adding information.

Each value records its source in `gdp_source` / `population_source`, so any figure can be traced. Some pair features are also missing in Gravity: common language for some territories (e.g. Guam, Curaçao), and colonial ties for China–Hong Kong, China–Taiwan and Israel–Palestine. These follow the same rule and stay missing.

In [ ]:
from exportspredictor import gravity
from exportspredictor.paths import data_dir

if not FIXTURE_MODE:
    print(gravity.build())  # downloads once, then rebuilds the derived tables (about 40 s)
    derived = data_dir() / "derived"
    in_flows = pl.concat(
        [flows.select(baci_iso3=pl.col(side).cast(pl.String)) for side in ["exporter", "importer"]]
    ).unique().collect()
    # Only countries that appear in the 2012-2024 Flows matter; BACI's file also lists retired codes.
    country_map = pl.read_parquet(derived / "country_map.parquet").join(in_flows, on="baci_iso3", how="semi")
    country_year = pl.read_parquet(derived / "country_year.parquet").filter(pl.col("iso3").is_in(country_map["iso3"].implode()))
    print("Exporters and Importers in the Flows:", country_map["baci_iso3"].n_unique())
    print("... with no Gravity pair features:", gravity.unresolved(country_map).height)
    print("... with no WDI data:", country_map.filter(~pl.col("in_wdi")).height)
    display(country_map.filter((pl.col("baci_iso3") != pl.col("iso3")) | pl.col("gravity_proxy").is_not_null()))

Where each GDP and population value comes from, and the countries that stay missing:

In [ ]:
if not FIXTURE_MODE:
    display(country_year.group_by("gdp_source").len().sort("len", descending=True))
    display(country_year.group_by("population_source").len().sort("len", descending=True))
    display(country_year.filter(pl.col("gdp_usd").is_null()).group_by("iso3").len().sort("iso3"))

### Data quality: BACI's UK totals against HMRC OTS 2023

BACI's UK Flows are not HMRC's statistics. BACI blends the UK's export report with each Market's import report (converted from CIF to FOB, weighted by how reliable each reporter has been), recasts every year to HS 2012, and works in USD. HMRC's **Overseas Trade Statistics** (OTS) are what the UK itself declared, in £, in that year's Combined Nomenclature (HS 2022 at six digits for 2023), and they exclude goods in transit and customs-warehouse re-exports (`docs/research/baci-hs12-feasibility.md`, section 4). Before trusting BACI as the target, we measure how far the two differ. This is the course's **data quality audit** with an independent primary source, which the notes do not cover.

`hmrc.build()` (a step in `scripts/rebuild.py`) downloads HMRC's 2023 bulk export archive (one fixed-width record per CN8 × country × port × month, 3.2 M records) and the UN Statistics Division's HS 2022 → HS 2012 conversion table. It sums ports and months to HS6 × Market, converts the codes and maps HMRC's country codes to ISO3. Value HMRC does not attribute to a Product (low-value consignments, suppressed commodity codes, chapter-level estimates) or to a Market (suppressed or unspecified countries) is kept with a missing key, so the totals still add up. `hmrc.crosscheck` then compares the result with BACI's UK Flows of 2023 in USD, at the Bank of England's 2023 average rate ($1.2434 per £).

In [ ]:
if FIXTURE_MODE:
    print("Fixture mode: the HMRC cross-check needs the real downloads (FIXTURE_MODE = False).")
else:
    from exportspredictor import hmrc

    hmrc.build()  # downloads once (36 MB), then parses and allocates in a few seconds
    hmrc_check = hmrc.crosscheck(
        hmrc.load(), hmrc.baci_uk_exports(flows, country_map)
    )
    print(hmrc_check["text"])
    display(pl.DataFrame([{key: value for key, value in hmrc_check.items() if key.endswith("_usd")}]))
    display(pl.DataFrame(hmrc_check["by_sector"]))
    display(pl.DataFrame(hmrc_check["by_market"]))
    print(hmrc_check["pairs"])

**The size of the difference.** For 2023, HMRC records **$528.6bn** of UK goods exports and BACI **$443.0bn**: HMRC is **$85.7bn, or 19%, higher** (BACI is 16% below HMRC). Almost all of that has a clear source:

| Part of the difference | USD bn | Why |
|---|---|---|
| Not attributed to a Product | 27.2 | HMRC books low-value consignments to a single code (9950 0000, $18.8bn), moves suppressed commodity codes to non-trade codes ($7.2bn), and estimates some trade only by chapter. BACI holds Flows only at HS6, so this value cannot appear in it. |
| Not attributed to a Market | 3.4 | Country suppressed for confidentiality (code "YY"), ships' stores and similar. |
| Precious metals (chapter 71) | 36.8 | Mostly non-monetary gold (7108): $65.2bn in HMRC against $33.1bn in BACI, almost all to Switzerland (+$28.2bn by Market). London's bullion trade is declared by the UK, but BACI reconciles it with partner reports that record it differently. |
| Everything else | 18.3 | About 5% of the remaining trade. Machinery (84, +$9.3bn), electrical goods (85, +$4.5bn) and organic chemicals (29, +$3.1bn) are higher in HMRC; inorganic chemicals (28, −$1.7bn), beverages (22, −$1.4bn) and ships (89, −$1.2bn) are higher in BACI. Plausible reasons are the CIF→FOB adjustment of partner reports, HMRC's monthly exchange rates against one annual average, and the HS 2022 → HS 2012 conversion, which moves value between Products. |

**At pair level the sources agree on the shape of UK trade.** 219,668 Product–Market pairs appear in both. Their values have a rank correlation of **0.89**, and 55% of BACI's value in those pairs lies within 25% of HMRC's figure. BACI also has 108,006 pairs that HMRC does not attribute at HS6 × Market level (most of them small: value HMRC suppressed, booked as low-value or never declared, recovered by BACI from the partner's import report), and HMRC has 1,739 that BACI lacks.

**Conclusion.** Every Exporter and Importer in the Flows maps to an ISO3 code with gravity features, and missing GDP and population follow one written rule with the source of each value recorded. Against HMRC, BACI understates UK goods exports by about a sixth in total, but for explained reasons: value HMRC does not attribute to a Product or Market, and gold. Outside those, the two differ by about 5% and rank pairs alike. So BACI is a sound target for the *shape* of UK trade, which is what the rankings use. Two cautions carry forward to the model card: chapter 71 (precious metals) is measured very differently by the two sources, and Export Gaps for small pairs inherit BACI's partner-report reconstruction.

## 3. Policy features: Trade Agreements and Trading Blocs

The policy features are the only columns a Scenario changes, so they must be right for every year. They come from the WTO's Regional Trade Agreements database (RTA-IS), "Export all RTAs" file. It lists every notified agreement with its goods entry-into-force date and **per-member entry and exit dates** (`docs/research/trade-agreement-coverage.md`). CEPII Gravity, the usual source in trade economics, stops in 2021, so it misses UK–Australia, UK–New Zealand and the UK's CPTPP accession.

For each pair and year, we build:

- `trade_agreement`: a goods Trade Agreement is in force between the two countries;
- `customs_union`: the agreement is a Customs Union (WTO type "CU");
- `eu_o/_d`, `cptpp_o/_d`, `efta_o/_d`: Trading Bloc membership of the Exporter and the Importer.

Each rule below follows how the WTO and Gravity code agreements:

- **The 1 July rule** (Gravity's): a membership counts in a year if it began before 1 July and had not ended by then. EU–Japan (1 Feb 2019) counts from 2019, and UK–Australia and UK–NZ (31 May 2023) from 2023. The UK's CPTPP accession (15 Dec 2024) counts only from 2025, so the 2024 Baseline does not include it.
- **A bloc that is a party is expanded to its members at each date.** "EU – United Kingdom" lists only "European Union" as the EU side. Croatia is still covered, because it is an EU member. An "EU – X" agreement links the EU side with X, not X's partners with each other.
- **The UK in 2020:** the UK left the EU (the Trading Bloc) on 31 Jan 2020. It stayed in the EU Customs Union and the EU's agreements until 31 Dec 2020 (the transition period). So for 2020 the UK's `eu_o` is 0, while its `customs_union` with EU members is still 1. The "EU Customs Union only" Scenario relies on this distinction.
- **Partial-scope agreements** (WTO type "PSA", such as the GSTP among developing countries) cut tariffs on short product lists only, so they do not count as Trade Agreements. Gravity's `fta_wto` leaves them out too.

This applies the course's **feature engineering** with explicit **data lineage**. Every flag traces back to a dated record in a primary source. `python -m exportspredictor.policy` rebuilds the tables from scratch; it is a step in `scripts/rebuild.py`.

In [ ]:
if FIXTURE_MODE:
    print("Fixture mode: the fixture world carries its own trade_agreement column (section 4).")
else:
    policy_files = policy.build()  # downloads the WTO export once, then rebuilds in seconds
    agreements = pl.read_parquet(policy_files["agreements"])
    trading_blocs = pl.read_parquet(policy_files["trading_blocs"])
    print(agreements.height, "pair-years with a Trade Agreement;", trading_blocs.height, "country-years in a tracked Trading Bloc")

The UK's policy features for a few Markets show the known dates:

- the EU Customs Union until 2020;
- the UK–EU agreement from 2021;
- Australia and New Zealand from 2023;
- no CPTPP membership yet in 2024.

In [ ]:
if not FIXTURE_MODE:
    uk_rows = pl.DataFrame(
        {"exporter": "GBR", "importer": ["FRA", "JPN", "AUS", "NZL", "MEX", "USA"]}
    ).join(pl.DataFrame({"year": list(range(2018, 2025))}, schema={"year": pl.Int32}), how="cross")
    uk_policy = policy.attach_policy(uk_rows, agreements, trading_blocs)
    display(uk_policy.pivot(on="year", index="importer", values="trade_agreement"))
    display(uk_policy.pivot(on="year", index="importer", values="customs_union"))
    display(uk_policy.filter(pl.col("importer") == "FRA").select("year", "eu_o", "cptpp_o", "efta_o"))

### Cross-check against CEPII Gravity, 2012–2021

Gravity V202211 codes `fta_wto` (per pair) and `eu_o/_d` (per country) up to 2021. It uses the same WTO database, with corrections by Thierry Mayer. We compare every pair-year that Gravity has. Most pairs have no agreement in either source, so the plain agreement rate flatters us. We therefore also report `positive_overlap`: of the pair-years that either source codes as in an agreement, the share that both do (the Jaccard index).

In [ ]:
if not FIXTURE_MODE:
    gravity_policy = policy.read_gravity_policy()  # from the gravity step's extract (section 2)
    crosscheck, disagreements = policy.gravity_crosscheck(agreements, trading_blocs, gravity_policy)
    display(crosscheck)

These are the results on the WTO export downloaded on 16 Nov 2026:

- **Trade Agreement:** the agreement rate is 95.5% over 549,900 pair-years, and the positive overlap is 70.6%. Both sources code 58,938 pair-years; 21,426 are ours only and 3,162 are Gravity only.
- **EU membership:** the agreement rate is 99.96%. The one difference is Croatia in 2013. Croatia joined on 1 July 2013, which is not *before* 1 July, so here it counts from 2014. Gravity counts any accession during the year.

Most of the **ours only** pair-years have a clear source:

- **Countries that Gravity never codes in any agreement (14%).** Gravity has no agreement at all for Montenegro, Bermuda or Yemen, and almost none for Liechtenstein. Yet the WTO lists, for example, EU–Montenegro (2008) and the Pan-Arab FTA.
- **Romania before 2018 (5%).** Gravity codes Romania's EU agreements only from 2018, although Romania joined the EU in 2007. This is a gap in Gravity.
- **The African Continental FTA (22%).** The WTO records it in force from 30 May 2019. Gravity does not code it, perhaps because trading under it began only in 2021.
- **The EU's Economic Partnership Agreements with African and Caribbean states (20%):** Eastern and Southern Africa, SADC, CARIFORUM and Ghana. They are notified to the WTO and applied, but Gravity's corrected `fta_wto` leaves most of these pairs out.
- **The rest (39%):** mostly COMESA accessions, SADC, EU–Serbia and EU–Central America. Here Gravity's corrections drop members that the export lists without dates.

Most of the **Gravity only** pair-years also have a clear source:

- **Territories missing from the WTO member lists (17%).** Mayotte, Réunion and other French overseas regions are part of the EU, but the WTO lists only France.
- **Haiti (17%).** Haiti signed the CARIFORUM agreement but has not ratified it. The WTO gives its entry date as 9999.
- **Latin American agreements notified as partial scope under LAIA**, such as Argentina–Bolivia and Argentina–Chile. We exclude these because they are PSAs.

The changes that the Scenarios and the Event Backtest rely on agree in both sources: EU–Japan from 2019, the UK's EU Customs Union to 2020, UK–EU and UK–Japan in 2021. Japan's only disagreements are CPTPP with Canada and New Zealand (in force 30 Dec 2018) and Japan–Mongolia (2016), which Gravity does not code.

UK pairs account for 368 of the disagreeing pair-years: 252 are ours only and 116 Gravity only. They are of the kinds above: Romania, Montenegro and Liechtenstein, the EPA partners, and a year's difference in when EU–Central America started. One more kind appears only for the UK. In 2021, Gravity keeps the UK in EU agreements that the UK did not roll over at the start of 2021, for example with Algeria, Syria, Andorra and the overseas territories. The WTO lists no UK agreement with these partners.

In [ ]:
if not FIXTURE_MODE:
    display(
        disagreements.filter(pl.col("feature") == "trade_agreement")
        .explode("years")
        .group_by("ours", "gravity")
        .agg(pair_years=pl.len(), countries=pl.concat_list("exporter", "importer").explode().n_unique())
    )
    display(disagreements.filter((pl.col("exporter") == "GBR") | (pl.col("importer") == "GBR")).head(20))

**Conclusion.** The policy features come from dated primary records, follow the same 1 July rule as CEPII Gravity, and agree with Gravity on every agreement the Scenarios and Event Backtests rely on. Where the two sources differ, the reasons are known and mostly gaps in Gravity. These columns are the only ones a Scenario changes (section 6), so their correctness bounds the meaning of every Scenario Effect.

## 4. The feature table

Every model input is computed in one place, `exportspredictor.features`, which the notebook (training) and the API (serving and Scenarios) both import, so a feature cannot drift between the two. This is the course's **feature engineering**, built under explicit **leakage** rules.

| Group | Features | What it captures |
|---|---|---|
| Gravity | log GDP and population of both countries, log distance, contiguity, common language, colonial tie, common colonizer | size and closeness of the two countries (section 2) |
| Supply | `log_supply_loo`: the Exporter's exports of the Product to all *other* Importers; `rca_loo`: its revealed comparative advantage in the Product | what the Exporter makes and sells well |
| Demand | `log_demand_loo`: the Importer's imports of the Product from all *other* Exporters; `high_income_share_loo`: the share of those from high-income Exporters | what the Market buys, and from whom ("Romania buys medicines from developed economies") |
| Product | `sector`: the HS2 chapter | Sector-level differences |
| Policy | Trade Agreement, Customs Union, EU, CPTPP and EFTA membership of each side (section 3) | the only columns a Scenario changes; monotone-constrained (ADR 0002) |

### Leakage rules

- **Leave-one-out supply and demand.** A row's own Flow is subtracted from every aggregate it feeds. For a UK row, UK supply of the Product excludes the target Market (ADR 0001). Otherwise the feature would contain the answer, and the out-of-fold model would copy Actual Exports back, shrinking every Export Gap. Revealed comparative advantage also leaves the row's Flow out of the Exporter's and the world's totals.
- **What leave-one-out does not remove: the UK inside other rows' demand.** Leave-one-out is per row, so a *non-UK* row's demand features (`log_demand_loo`, `high_income_share_loo`) leave out that row's Exporter but still include the UK's Flow to the same Market. France's whisky exports to Japan, for example, carry Japan's whisky imports from everyone but France, UK whisky included, and the fold model that holds out UK–Japan whisky still trains on that row. This means held-out UK Flows are not perfectly invisible: no fold model can read a held-out pair's Actual Exports from its own features or label, so Export Gaps are not shrunk by memorisation, but where the UK supplies a large share of a Market's imports of a Product, the non-UK rows teach the model a demand level that includes UK sales, which can pull that pair's Export Potential slightly towards its Actual Exports. ADR 0001 records this as an accepted consequence: closing it would mean rebuilding every non-UK row's demand features without the UK, once per fold.
- **No features from the pair's trade in other Products.** Total UK–Japan trade, for example, is left out on purpose. It is high largely *because* of the policy conditions between the two countries, so it would absorb the policy effect. A Scenario that flips only the policy columns would then understate the Scenario Effect.
- **Training years only.** The clip ranges and the Sector encoder (`FeatureTransform`) are fitted on training years. Test-year data cannot reach them, and serving inputs are clipped to the range seen in training (as tariffs would be, ADR 0002). Aggregates are computed within each year, so a year's features never use a later year's Flows.

The test suite checks each rule (`tests/test_features.py`). It changes the UK–Japan whisky Flow and confirms that this row's supply and demand features stay the same, while the same Product's UK–Canada row changes. It moves the UK's other-Product exports from Japan to Canada and confirms that no UK–Japan whisky feature moves. And it distorts the test years' Flows a thousandfold and confirms that the fitted transform is unchanged.

### Training rows: non-zero Flows plus sampled relevant zeros

BACI records only non-zero Flows, but a model of Export Potential must also learn where trade does *not* happen. The full Exporter × Importer × Product × year grid has about 3.45 bn cells, too many to train on, and most of it is uninformative (Malta not exporting jet engines). So the training rows are:

- **every non-zero Flow**, with weight 1;
- **a random sample of relevant zero cells**, with weight 1/rate. A cell is a relevant zero when the Exporter exports the Product to someone and the Importer imports it from someone that year, but not from each other. These are the zeros that carry information: both sides are active in the Product, and yet no trade happens between them.

On BACI there are about 95 M relevant zero cells a year against 9–12 M non-zero Flows. The default rate of 2% keeps about 1.9 M zeros a year, roughly one for every six non-zero Flows. Each kept zero carries weight 50, so weighted totals stay unbiased: this is the Horvitz–Thompson estimator, which the fixture test checks against an exact count. This goes beyond the course's treatment of class imbalance. Instead of rebalancing classes, we subsample the majority and reweight it, so the Tweedie loss still sees zeros in their true proportion. Whether a cell is kept depends only on a hash of its keys and a seed, so the sample is reproducible and never shifts when other data change. The rate and seed are recorded in the artefact metadata. The rate is fixed at 2% rather than tuned: the hyperparameter search in section 5 is kept small so that the real run stays within a few hours.

### The UK grid for scoring

For scoring, the UK's grid of every Product × Market is built in full for each year, with no sampling: about 5,200 Products × 225 Markets ≈ 1.17 M pairs. About 72% of them have no Actual Exports. Each pair's Export Potential comes from its out-of-fold model.

On real data, `features.build()` writes one file per year for the training rows and one for the UK grid (`derived/features_V202601/`). It builds 2012–2024 in about 90 s, with a measured peak memory of 7.9 GB, well inside 32 GB, because each year is built from that year's Flows alone.

In [ ]:
from exportspredictor import features

if FIXTURE_MODE:
    feature_world = build_world()
    feature_table = features.build_features(
        features.training_rows(feature_world.flows, rate=0.5), feature_world.flows, feature_world.sources
    ).lazy()
else:
    feature_dir = features.build()  # about 90 s for 2012-2024, peak memory about 8 GB
    feature_table = pl.scan_parquet(feature_dir / "training_*.parquet")

is_zero = pl.col("value_usd") == 0
feature_table.group_by("year").agg(
    rows=pl.len(),
    sampled_zeros=is_zero.sum(),
    relevant_zeros_estimated=pl.col("weight").filter(is_zero).sum(),
).sort("year").collect()

On real data, 2023 has 11.8 M non-zero Flows and 1.89 M sampled zeros (13.6 M rows). The weights estimate about 94.5 M relevant zero cells. Across 2012–2024 there are about 165 M training rows, about 3.3 GB of Parquet.

The features of one UK row, UK whisky to Japan in the latest year:

In [ ]:
feature_table.filter(
    (pl.col("exporter") == "GBR") & (pl.col("importer") == "JPN") & (pl.col("hs6").cast(pl.String) == "220830")
).sort("year").tail(1).collect().transpose(include_header=True, column_names=["value"])

**Conclusion.** One package function builds every model input for training and serving alike, which removes training/serving skew by construction (course: *training/serving skew and feature stores*). The leakage rules are enforced by tests rather than by care: leave-one-out supply and demand stop a row from seeing its own Flow, and leaving out the pair's trade in other Products keeps the policy effect in the policy columns. Sampled, reweighted zeros let a model of 165 M rows learn where trade does not happen without building the 3.45 bn-cell grid.

## 5. Training: tuning, out-of-fold fold models and the Baseline

The model is a single gradient-boosted tree ensemble (XGBoost, ADR 0002), trained on the GPU (`device="cuda"`). It extends the course's tree-ensemble material in four ways:

- **Tweedie loss.** Flows are non-negative, often zero and extremely skewed, so squared error on raw values would chase a few huge Flows. The Tweedie objective models a log-mean with a variance that grows with the mean, which suits exactly that (course: skewed targets and loss choice).
- **Monotone constraints** (+1 on every policy feature, ADR 0002): adding a Trade Agreement, Customs Union or Trading Bloc membership never lowers Export Potential. The model *associates* an agreement with higher Flows; it does not show cause.
- **Out-of-fold cross-fitting for the UK** (ADR 0001, beyond the course's k-fold cross-validation). UK Product–Market pairs are split into 5 folds by a hash of the pair, so every year of a pair shares a fold. Fold model *k* trains on all non-UK rows plus the UK pairs outside fold *k*, and scores only fold *k*. A pair's Export Potential therefore never comes from a model that saw its Actual Exports, so the Export Gap is not shrunk by memorisation. `tests/test_folds.py` checks this by distorting one fold's UK labels a thousandfold: that fold's scores do not move at all, while every other fold's do.
- **Training data that does not fit in memory.** The ~150 M training rows of 2012–2023 are about 16 GB as floats. They are streamed to XGBoost one year at a time through a data iterator into a `QuantileDMatrix`, which stores only the binned features. Even binned, all of them fill the RTX 3080's 10 GB once training starts, so the real run keeps a hash sample of 30% of *non-UK* rows, reweighted by 1/0.3 (the same Horvitz–Thompson idea as the sampled zeros in section 4). Every UK row is kept.

### The time split and the search

The years are split as the course recommends for data with a time order (course: train/validation/test splits and leakage):

| Role | Years | Used for |
|---|---|---|
| training (search) | 2012–2021 | fit each candidate configuration |
| validation | 2022 | early stopping and choosing the configuration |
| test | 2023 | the evaluation (section 7); never read here |
| refit | 2012–2023 | the final fold models that score the Baseline |
| Gap Period | 2022–2024 | each year scored by the refit fold models; the Export Gap is the mean of the three yearly gaps |
| Baseline | 2024 (provisional) | the starting point of every Scenario and Blocked Opportunity value |

The search is deliberately small, five configurations: tree depth 6, 8 or 10, and Tweedie variance power 1.3, 1.5 or 1.7 at depth 8. Each is trained with early stopping on 2022 (course: early stopping as regularisation). Candidates with different variance powers optimise different losses, so they are all compared on one fixed metric, the weighted Tweedie negative log-likelihood at power 1.5. The feature transform (clip ranges and Sector encoder) is fitted on the search's training years for the search, and on 2012–2023 for the refit, never on a year it is scored on.

In [ ]:
if FIXTURE_MODE:
    world = build_world()
    bundle = pipeline.build_artefacts(world, source="fixture")
else:
    bundle = pipeline.build_real_artefacts()  # about an hour on an RTX 3080, evaluation included
{key: bundle.metadata[key] for key in ("source", "device", "params", "rounds", "tuning_seconds", "training_seconds", "non_uk_sampling_rate")}

The search results on the 2022 validation year, best first (lower is better):

In [ ]:
tuning = pl.DataFrame(bundle.metadata["tuning"]).unnest("config")
tuning

On the real data (run of 17 Nov 2026, artefact version `20261117T002435Z`), the search took 8.3 minutes on the RTX 3080 and the refit of the five fold models 20.5 minutes, about 30 minutes in all. Both times are recorded in the artefact metadata.

| Depth | Variance power | Rounds (early stopping) | Validation Tweedie NLL @1.5 |
|---|---|---|---|
| 8 | 1.3 | 1,491 | **341.5** |
| 10 | 1.5 | 420 | 345.6 |
| 8 | 1.5 | 875 | 346.4 |
| 6 | 1.5 | 1,481 | 349.3 |
| 8 | 1.7 | 400 | 353.5 |

The differences are small (about 3% between best and worst), so the model is not sensitive to these choices. A lower variance power wins: it treats large Flows as less noisy, so the model works harder to fit them, and large Flows dominate the Export Gaps Advisors read. The winning configuration stopped at 1,491 of the 1,500 allowed rounds, so a longer budget might gain a little more. We keep the cap so that the run stays short. Whether the tuned model beats a naive baseline on the untouched 2023 test year is the job of the evaluation section.

### The Baseline table and the Gap Period

The refit fold models score the UK grid of each **Gap Period** year, 2022, 2023 and 2024, each pair by the fold model that held it out (ADR 0001), each year with that year's features. The grid is the union of the three years' grids, so a pair is scored in every year even if its Product was not traded in one of them, and a year with no Flow counts as $0 of Actual Exports. The **Export Gap** is the plain mean of the three yearly gaps (Export Potential minus Actual Exports), so gaps still add up across pairs. The Baseline table holds the three Gap Period means and, separately, the 2024 Export Potential: Scenarios and Blocked Opportunity values start from 2024 alone.

Why average? One year's Actual Exports are noisy: a single large shipment, or a re-routed contract, can move a pair's Flow several-fold from one year to the next, and the 2024 figures are provisional. Averaging three yearly gaps cuts that year-to-year variance, the same idea as bagging averaging many noisy trees (course: *ensembles: bagging averages to cut variance*), while keeping the gap recent. Why 2022–2024 and not five years? **2020 and 2021 are left out on purpose**: COVID-19 collapsed and then rebounded trade, and the end of the Brexit transition period (January 2021) reset UK–EU trade, so those years describe shocks rather than the UK's current fundamentals. Using them would bias the gaps of exactly the Markets Advisors care most about.

The artefacts also store each pair's 2024 model inputs and fold for the Scenarios and explanations to come, and a yearly history: Export Potential, Actual Exports and Export Gap for each Gap Period year, and the UK's Actual Exports for 2019–2021.

In [ ]:
baseline = bundle.baseline.join(bundle.products, on="hs6", how="left").join(
    bundle.markets.rename({"name": "market"}), on="iso3", how="left"
)
baseline.select(
    pairs=pl.len(),
    products=pl.col("hs6").n_unique(),
    markets=pl.col("iso3").n_unique(),
    pairs_with_exports=(pl.col("actual_exports_usd") > 0).sum(),
    actual_exports_usd=pl.col("actual_exports_usd").sum(),
    export_potential_usd=pl.col("export_potential_usd").sum(),
    baseline_export_potential_usd=pl.col("baseline_export_potential_usd").sum(),
    opportunities=(pl.col("export_gap_usd") > 0).sum(),
)

How one pair's Export Gap is formed: whisky (`220830`) to Japan, year by year. The mean of the three yearly gaps is the Export Gap in the Baseline table.

In [ ]:
whisky_japan = (pl.col("hs6") == "220830") & (pl.col("iso3") == "JPN")
display(bundle.history.filter(whisky_japan, pl.col("export_gap_usd").is_not_null()))
bundle.baseline.filter(whisky_japan).select(
    "export_potential_usd", "actual_exports_usd", "export_gap_usd", "baseline_export_potential_usd"
)

### Sanity check: the top Opportunities

A model can score well and still rank nonsense, so we read the largest Export Gaps and ask whether a trade economist would find them plausible.

In [ ]:
baseline.sort("export_gap_usd", descending=True).select(
    "hs6", "description", "market", "export_potential_usd", "actual_exports_usd", "export_gap_usd"
).head(15)

One Product an Advisor would know well, whisky (`220830`): the Markets with the largest Export Gaps, and the ones where the UK already exceeds its Export Potential.

In [ ]:
whisky = baseline.filter(pl.col("hs6") == "220830").sort("export_gap_usd", descending=True)
whisky_columns = ["market", "export_potential_usd", "actual_exports_usd", "export_gap_usd"]
display(whisky.select(whisky_columns).head(8))
whisky.select(whisky_columns).tail(5)

These figures are from the first version of the model, which took the Export Gap from 2024 alone; they are updated when the model is rebuilt with the Gap Period. On the real data, the Baseline covers all 1,169,550 UK Product × Market pairs of 2024 (5,198 Products × 225 Markets), of which 306,102 have Actual Exports, together $407 bn. Read the results with these points in mind:

- **Export Potential totals $692 bn, and 95% of pairs have a positive Export Gap.** This is what a conditional mean does: a pair with no exports but plausible fundamentals gets a small positive Export Potential, and there are 860k such pairs. So individual small gaps are noise, and only the *size* ranking is informative. On pairs with exports, the Spearman rank correlation between Export Potential and Actual Exports is 0.69, so the model reproduces the shape of UK trade without copying it (the pairs were held out).
- **The top Opportunities are plausible.** They are concentrated in the UK's known strengths: medicines to China and the Netherlands; aircraft parts to Italy, Singapore, China and France; jet engines and their parts to Hong Kong, France and Poland; cars and tractors to Germany. These are large Markets for Products the UK makes well, where comparable Exporters sell more.
- **Some top gaps have explanations the model cannot see.** Crude oil to Ireland and gas to France and China depend on pipelines, refineries and long-term contracts, not on the fundamentals in our features.
- **Sanctions are invisible to the model.** For whisky, Hong Kong leads, then the **Russian Federation**: UK whisky exports there fell from $211 M in 2021 to $19 M in 2022 under the sanctions (the history table shows it), but no feature records sanctions. So an Advisor would read this row as a Barrier the model does not know about, not as an Opportunity. This is a known limitation for the Model Evaluation page.
- **Whisky's negative gaps are where the UK already over-performs**: the USA, Singapore, Taiwan, France and India, its long-established whisky Markets.

As always, these are associations: the model says that comparable Exporters sell more of these Products to these Markets, not that the UK would if it tried.

**Conclusion.** A small, honest search (five configurations, early stopping on 2022) chose a depth-8 model with Tweedie variance power 1.3, and the five out-of-fold fold models score every UK pair in each Gap Period year without having seen it; the Export Gap is the 2022–2024 mean, which steadies the ranking against one-off shipments while leaving out the COVID and Brexit-transition years. The Baseline's top Opportunities are plausible UK strengths in large Markets; the exceptions (pipelines, sanctions) are things no feature records. Two training compromises go into the model card: the 30% sample of other Exporters' rows, forced by GPU memory, and the round cap the best configuration nearly reached.

## 6. Scenarios and Blocked Opportunities

A Scenario asks the trained model a what-if question: copy the 2024 Baseline rows of the affected Markets, change **only the policy columns**, and re-predict each pair with the same fold model that scored its Baseline (ADR 0001). The Scenario Effect is the difference. `exportspredictor.scenarios` does this for the notebook and the API alike, and checks on every run that no non-policy column moved (a leakage guard: a Scenario that also shifted, say, demand would report the demand change as a policy effect).

This extends two course concepts:

- **Ensembles: gradient boosting.** The fold models carry a +1 monotone constraint on every policy feature (ADR 0002), so switching a Trade Agreement on can never lower a prediction. A property-based test checks this over the fixture rows with random policy settings. Without the constraint, a tree could learn a spurious dip and a Scenario could show a negative effect from adding an agreement.
- **Correlation, multicollinearity and causation.** Flipping one feature and re-predicting is the same move as the course's counterfactual fairness test, but the answer is still learned from correlations in past trade. So we say "the model associates the CPTPP with +X%", never "joining CPTPP causes +X%". The Event Backtests of section 7 test these effects against real agreements.

The four Trading Bloc presets (wayfinder #8) change only the rows of the bloc's member Markets in the Baseline year:

| Preset | Policy columns set on member Markets' rows |
|---|---|
| EU re-join | `trade_agreement`, `customs_union`, `eu_o` (as for any pair of EU members in training) |
| EU Customs Union only | `customs_union` |
| CPTPP | `trade_agreement`, `cptpp_o`, with accession treated as complete |
| Leave | for each bloc the UK is in: its `<bloc>_o`, `trade_agreement` and `customs_union` switched off |

Because a membership counts in a year only if it began before 1 July, the 2024 Baseline has no UK CPTPP membership (accession took effect from December 2024), so the CPTPP preset is a live prediction that 2025 data can check. In 2024 the UK is in no tracked Trading Bloc, so Leave changes nothing, and the API says so.

In [ ]:
from exportspredictor import scenarios

preset_results = {
    name: scenarios.run(bundle.baseline_features, bundle.baseline, bundle.fold_models, scenarios.Scenario(preset=name))
    for name in scenarios.PRESETS
}
pl.DataFrame(
    [
        {
            "preset": scenarios.PRESETS[name].title,
            "pairs_changed": result.pairs.height,
            **result.total(),
            "blocked_opportunities": result.blocked_opportunities().height,
        }
        for name, result in preset_results.items()
    ]
)

On the real artefact (version `20261117T003531Z`), on CPU:

| Preset | Pairs changed | Scenario Effect | Share of Baseline | Blocked Opportunities (≥20%, ≥$1M) |
|---|---|---|---|---|
| EU re-join | 140,346 (27 Markets) | +$151.3bn | +44.7% | 13,458 |
| EU Customs Union only | 140,346 | +$58.4bn | +17.3% | 3,285 |
| CPTPP | 57,178 (11 Markets) | +$5.7bn | +8.1% | 422 |
| Leave | 0 | 0 | | 0 |

Each bloc Scenario returns in about a second through the API. The EU figures are large because the model associates intra-EU membership with much denser trade than a free-trade agreement alone; they describe what the model has learned from EU members' trade, not what re-joining would bring. The CPTPP effect is smaller: the UK already has agreements with 9 of the 11 members, so for those nine only the CPTPP flag changes, and Brunei and Malaysia also gain an agreement. The CPTPP effect per Market:

In [ ]:
preset_results["cptpp"].by_market().join(bundle.markets.select("iso3", "name"), on="iso3", how="left")

### Precomputed Blocked Opportunity values

The rankings can also be ordered by **Blocked Opportunity value**: for every pair with no Trade Agreement in 2024, the Scenario Effect of adding one, re-predicted by the pair's fold model when the artefacts are built (`scenarios.blocked_effects`). It is 0 where an agreement already exists. A pair is a Blocked Opportunity when that effect is at least 20% of its Baseline Export Potential and at least $1M. The largest values:

In [ ]:
baseline.with_columns(
    blocked_opportunity=scenarios.is_blocked("blocked_effect_usd", "export_potential_usd")
).sort("blocked_effect_usd", descending=True).select(
    "hs6", "description", "market", "export_potential_usd", "blocked_effect_usd", "blocked_opportunity"
).head(10)

### What the monotone constraints change

The +1 monotone constraint on every policy feature (ADR 0002) is a design choice, so we measure what it does rather than only state it. `constraints.compare` trains two models that differ *only* in the constraint, with the same rows, tuned parameters, round count and seed, and asks both the same Scenario questions for the same UK pairs: add a Trade Agreement where there is none, and remove it (with any Customs Union) where there is one. It counts the pairs that move the "wrong" way, an agreement that lowers Export Potential or a removal that raises it, and compares the size of the effects. This extends the course's gradient-boosting material: a monotone constraint is a form of domain knowledge built into the trees, a kind of regularisation that trades a little fit for answers that respect a known direction.

Both models follow ADR 0001: they train on all non-UK rows and the UK pairs outside one fold, and score only that fold's pairs, so no effect comes from a model that saw the pair. To keep the real run cheap (two models rather than ten fold-sized ones), the real comparison trains on three Sectors (22 beverages, 30 pharmaceuticals, 87 vehicles) and one held-out fold, with the tuned parameters and round count. The fixture run uses every Sector and fold.

In [ ]:
monotone = bundle.evaluation["monotone_constraints"]
print(monotone["text"])
pl.DataFrame(
    [{"model": m["model"], "change": change, **m[change]} for m in monotone["models"] for change in ("add", "remove")]
)

On the real data (artefact version `20261117T022514Z`, copied from `20261117T015239Z`; 70 s on the RTX 3080 for both models, 1,491 rounds each), over the held-out fold's UK pairs in the three Sectors:

| Model | Adding a Trade Agreement (3,327 pairs) | Removing one (2,317 pairs) | Total effect of adding |
|---|---|---|---|
| Constrained | 0 pairs lower; median **+22.8%** | 0 pairs higher; median −20.6% | **+56.4%** |
| Unconstrained | **869 pairs (26%) lower**, down to −47%; median +16.1% | **453 pairs (20%) higher**, up to +57%; median −23.0% | +34.5% |

The two models' Baselines rank these pairs almost identically (Spearman 0.974), so the constraint costs little fit. Without it, a quarter of the Blocked Opportunity questions would get an answer an Advisor could not defend: a Trade Agreement associated with *less* trade, sometimes by half. That is a spurious dip the trees learned from noise in past agreements, and it is what the constraint removes.

The comparison also shows the price, and we state it plainly. With the negative answers removed, the constrained model's total effect of adding an agreement is larger (+56% against +35%). The constraint does not make Scenario Effects more accurate; it makes their *direction* consistent. Their *size* is checked separately by the Event Backtests in section 7, which find that the model overstates real agreements several-fold. So the constraint is right for the question an Advisor asks ("would an agreement help here?"), and the size of each Scenario Effect stays an upper-end association, not a forecast.

**Conclusion.** Scenarios change only policy columns and re-predict each pair with its own fold model, and the monotone constraint guarantees that an agreement never lowers Export Potential; the comparison with an unconstrained model shows what that guarantee removes. The bloc presets give large associations (EU re-join +45% of the bloc's Baseline, CPTPP +8%). Section 7 shows that these associations are several times what a real agreement delivered, so they are an upper-end order of magnitude, never a forecast.

## 7. Evaluation: accuracy on the 2023 test year

The course's **time split** is the honest way to evaluate a model that will be used on later data: fit on the past, test on a year the model never saw. Here the years are train 2012–2021, validate 2022 (the hyperparameter search of section 5) and test 2023. The Baseline models of section 5 were refit on 2012–2023, so they cannot be tested on 2023. The evaluation therefore trains five **test-year fold models** on 2012–2022 with the configuration and round count chosen on 2022, and scores 2023 with them. Nothing about 2023 shaped tuning or training: the pipeline asserts that the test year is not among the tuning years, and a test (`test_the_test_year_is_untouched_during_tuning`) multiplies every 2023 Flow by 1,000 and checks that the search results do not move. This is the course's **leakage** rule applied to model selection, not only to preprocessing.

Following the course's **baselines and metrics** advice (report the naive baseline first), every metric is computed for the model and for a **naive baseline**: the Exporter's share of the Product's world exports times the Importer's imports of that Product. It is the "no fundamentals" prediction a trade analyst would make on a spreadsheet, and like the model's features it leaves the Flow itself out.

The course distinguishes the **loss** a model minimises from the **metric** people read. We report one of each kind and one that goes beyond the notes:

| Metric | Kind | Why |
|---|---|---|
| Tweedie deviance (power 1.5, weighted) | the training loss | measures the fit on every row, zeros included; sampled zeros count by their weight |
| RMSE of log(1 + v), non-zero Flows | the course's RMSE, on a log scale | the typical error in orders of magnitude; raw-dollar RMSE would be ruled by a handful of huge Flows |
| Spearman rank correlation within each Product | a ranking metric (beyond the notes) | Advisors read the model as a ranking of Markets for one Product, so we score exactly that, averaged over Products |

Two scopes are evaluated. **All Exporters** scores every 2023 training row (every non-zero Flow plus the sampled zero cells): UK rows out-of-fold, other rows by the mean of the five fold models. **UK out-of-fold** scores the UK's full 2023 Product × Market grid, each pair by the fold model that never saw it (ADR 0001), which is how the Baseline is produced. As in section 5, the models train on every UK row and a 30% reweighted sample of non-UK rows on real data. The fixture keeps every row.

The headline is the within-Product rank correlation for UK pairs, phrased for an Advisor:

In [ ]:
from exportspredictor import evaluation

report = bundle.evaluation
print(report["headline"]["text"])
{key: report[key] for key in ("test_year", "test_training_years", "validation_year", "non_uk_sampling_rate", "evaluation_seconds")}

Every metric, model against naive baseline, for both scopes:

In [ ]:
pl.DataFrame([{"scope": scope, **result["overall"]} for scope, result in report["accuracy"].items()]).drop("group", strict=False)

### Where the model is weak: Sectors and regions

The breakdowns are the course's *evaluating beyond the average: worst group*. They also feed the **bias review** of section 10. Here bias is about countries and Sectors, not people: BACI's data are weaker where countries report late or carelessly, and the model may be weaker for those Markets. Each table is sorted weakest first, by the model's within-Product rank correlation for UK pairs.

In [ ]:
pl.DataFrame(report["accuracy"]["uk_out_of_fold"]["by_sector"]).head(10)

In [ ]:
pl.DataFrame(report["accuracy"]["uk_out_of_fold"]["by_region"])

### Stability of the 2024 Baseline

BACI's 2024 data are provisional, so we check that the Baseline's rankings are not an artefact of one noisy year. The Baseline models also score the 2023 UK grid, and we compare the two years' rankings within each Product, and the overlap of the top 1,000 Opportunities. This is also the course's **drift** check in miniature: when BACI releases a new year, the same comparison shows how far the rankings moved.

In [ ]:
print(report["stability"]["text"])
report["stability"]

### Known limitations

These are shown on the app's Model Evaluation page, so that Advisors can caveat their briefings. The last one came out of section 5's sanity check of the top Opportunities.

In [ ]:
pl.DataFrame(evaluation.LIMITATIONS)

### Results on the real data

These are the results of the run of 17 Nov 2026 (artefact version `20261117T010658Z`). The five test-year fold models took 19.8 minutes to train on the RTX 3080, and scoring the 13.6 M test rows and the 1.17 M-pair UK grid took 4.6 minutes on the CPU.

| Scope | Metric | Model | Naive baseline |
|---|---|---|---|
| All Exporters (13.6 M rows) | Tweedie deviance | **312** | 5,136 |
| | RMSE of log(1 + v), non-zero Flows | **2.87** | 3.42 |
| | Spearman within Product | **0.71** | 0.56 |
| UK out-of-fold (1.17 M pairs) | Tweedie deviance | **658** | 3,653 |
| | RMSE of log(1 + v), non-zero Flows | 2.76 | **2.74** |
| | Spearman within Product (headline) | **0.54** | 0.52 |

How to read these numbers:

- **Across all Exporters the model clearly beats the naive baseline** on every metric. Its deviance is about a sixteenth of the baseline's, and its within-Product ranking improves from 0.56 to 0.71.
- **For UK pairs the gain in ranking is small.** The headline rank correlation is 0.54 against 0.52, and on RMSE of log value the naive baseline is marginally better (2.74 against 2.76). Two reasons are plausible:
  - Out-of-fold scoring removes exactly the pair-specific information (the UK's own past exports to that Market) that would make UK rankings easy. That is the point of ADR 0001.
  - The UK's world market share in a Product is already a strong signal for its own Markets.
- **The model is much better calibrated for UK pairs.** Its deviance is about a sixth of the baseline's, so Export Potential is a better estimate of the *size* of a Flow even where the ranking gain is small. The app's headline states the narrow margin rather than a bare "beats the baseline".
- **Weakest Sectors for UK pairs** are fish (chapter 03, rank correlation 0.24), live animals (01, 0.25), ores (26, 0.25), wood pulp (47, 0.29) and meat (02, 0.30). In each of these the naive baseline does slightly better. All are commodities where sanitary rules, quotas and logistics decide Markets, and the model sees none of these.
- **Weakest regions for UK pairs** are Latin America & Caribbean (0.33) and Sub-Saharan Africa (0.35). Small territories BACI leaves unclassified score worst (0.21). North America is the strongest (0.83). This matches the bias concern above: the model is weakest where Markets are small, distant or poorly reported, so Opportunities there need more checking.
- **Stability.** Scored by the same Baseline models, each Product's 2024 ranking of Markets has a rank correlation of 0.80 with its 2023 ranking. 62% of the top 1,000 Opportunities are the same in both years. Rankings are broadly stable, but the provisional 2024 data move the top of the list noticeably, so the top 1,000 should not be treated as fixed.

### Lift: do large Export Gaps point at growth?

Accuracy says how well the model reproduces Flows in a year it never saw. An Advisor needs something else: when the model says the UK underperforms in a pair, does that pair tend to grow later? We cannot test that on UK pairs with the Baseline models, because the future after 2024 has not happened. So we replay the past (`backtests.lift_backtest`):

1. **Hold out 20% of Exporters entirely**, chosen by a hash of their codes. They play the UK's role: just as each UK pair is scored by a fold model that never saw it (ADR 0001), these countries' Flows are never trained on.
2. **Train one model on 2012–2017** with the tuned configuration, round count and 30% non-UK sample of section 5.
3. **Score the held-out Exporters' pairs in 2017** and rank them by Export Gap.
4. **Look up 2023.** A pair *grew materially* if its Flow rose by at least 50% and at least $100,000 (so a pair that appears from nothing counts once it reaches $100,000). Sampled zero cells count by their weight.

**Lift** is the growth rate of the top tenth of pairs by Export Gap divided by the growth rate of all pairs. The course meets lift as a gate number for Grad-CAM ("share of attention on ink ÷ share of image that is ink", with a gate at lift ≥ 1); here it plays the same role for Opportunities: above 1 means the ranking concentrates growth. Two things go beyond the notes:

- **A confidence interval from a cluster bootstrap.** The course meets the bootstrap inside random forests. Here it resamples whole Exporters 1,000 times, because one country's pairs rise and fall together (a currency, a recession), so resampling pairs would overstate the precision.
- **A size-matched Lift.** Large gaps belong to large pairs, and a $100,000 rise is easier for a large pair. So we also compare the top tenth with pairs in the same Export Potential tenth. If that ratio stays above 1, the gap says something beyond size.

Lift is a tendency across many pairs: it does not promise that any one Opportunity will grow.

In [ ]:
lift = report["lift"]
print(lift["text"])
pl.DataFrame(lift["by_decile"]).rename({"decile": "Export Gap tenth (1 = largest)", "rate": "share that grew", "pairs": "weighted pairs"})

### Event Backtests: Scenario Effects against a real Trade Agreement

A Scenario Effect is an association learned from many agreements (ADR 0002), so it should be checked against one that really happened. The **EU–Japan Economic Partnership Agreement** came into force on 1 February 2019. `backtests.event_backtest` does the following:

1. **Train one model on 2012–2018 only.** It never reads a year from the event onwards: `train_backtest_model` refuses later years, and a test (`test_no_event_backtest_model_sees_data_from_after_its_event`) multiplies every Flow from the event year on by 1,000 and checks that the trained model is byte-for-byte the same.
2. **Predict 2019–2021** EU→Japan and Japan→EU Flows with the agreement on and off. This is the course's *counterfactual test* (flip one attribute, hold the rest), used here as a measurement, not a fairness check. The difference is the model's Scenario Effect.
3. **Measure what happened** with a **difference-in-differences**, a standard method from econometrics that is not in the course notes: the change in EU exports to Japan from 2016–2018 to each later year, divided by the change in EU exports to the **Control Markets** over the same years. The Control Markets are the ten largest high-income Markets that had no Trade Agreement with any EU member at any point in the window. A 95% interval comes from resampling Products.

**Caveat.** COVID-19 disrupted trade in 2020 and 2021, and not evenly across Markets, so 2019 is the cleanest year. The method assumes EU–Japan trade would otherwise have moved like EU trade with the control Markets, and values are nominal USD, so exchange-rate swings (the yen against the dollar) move them.

A secondary check uses the **UK–Australia and UK–New Zealand** agreements (in force 31 May 2023). The test-year fold models of this section were trained on 2012–2022, before them, so they score it, each UK pair by the model that held it out. Only 2024 serves as a post year, and 2024 is provisional.

In [ ]:
backtest = report["event_backtest"]
print(backtest["text"])
pl.DataFrame(
    [
        {
            "event": event["name"],
            "direction": direction["label"],
            "year": str(row["year"]),
            "predicted": row["predicted"],
            "observed": row["observed"],
            "observed_low": row["observed_ci"][0],
            "observed_high": row["observed_ci"][1],
        }
        for event in backtest["events"]
        for direction in event["directions"]
        for row in [*direction["by_year"], {**direction["pooled"], "year": "pooled"}]
    ]
)

In [ ]:
{(event["name"], direction["label"]): direction["controls"] for event in backtest["events"] for direction in event["directions"]}

### Backtest results on the real data

From the run of 17 Nov 2026 (artefact version `20261117T013014Z`, copied from `20261117T010658Z`). The Lift model (2012–2017, 45 of 226 Exporters held out) and the EU–Japan model (2012–2018) each took a few minutes on the RTX 3080, with the 1,491 rounds and 30% non-UK sample of section 5. They are cached under `data/derived/backtest_models/`, so a re-run (`python -m exportspredictor.backtests`) takes about 15 seconds.

**Lift.** The held-out Exporters have 1.27 M sampled 2017 pairs (17.0 M once sampled zeros are weighted). Of the top tenth by Export Gap, 4.5% grew materially by 2023, against 0.67% of all pairs: a **Lift of 6.6 (95% interval 4.8 to 10.4)**. The size-matched comparison changes the reading. Pairs in the same Export Potential tenth grew 5.4% of the time, so the **size-matched Lift is 0.83 (0.80 to 0.87)**. The largest gaps point at the pairs most likely to grow, but mostly because they are large pairs. Among pairs of similar size, a large gap was, if anything, a slightly weaker sign of growth. The growth share by gap tenth is 4.5%, then 0.3%, under 0.1% for the middle tenths, and 1.8% for the tenth with the most negative gaps (pairs already above their Export Potential, which are also large). For Advisors this means an Opportunity marks a Market worth a look, not one more likely to grow than its peers of the same size. The Model Evaluation page says so in those words.

**EU–Japan Event Backtest.** The model associates the agreement with **+45%** in EU Export Potential to Japan over 2019–2021 (+44%, +45%, +46% by year). The observed difference-in-differences against the Control Markets (USA, Russia, UAE, Australia, Saudi Arabia, Hong Kong, Taiwan, Qatar, Kuwait, New Zealand) is **+3% (95% interval −6% to +15%)**. It is +5% in 2019, the year COVID cannot explain. For Japan's exports to the EU the model says **+34%** and the observed change is **+0% (−9% to +10%)**. So the model's Scenario Effects are **several times larger than the change actually observed**, well outside the interval. The reasons are plausible but untested here:
- The model learned from all agreements in 2012–2018, many of them between countries whose trade was growing anyway.
- With no pair-level features in the model (a deliberate choice, see section 4), the agreement flag also absorbs closeness between partners.
- The EU–Japan agreement phased in tariff cuts over years, so its full effect lies beyond 2021.

**UK–Australia/New Zealand (secondary).** The test-year fold models say **+27%** for 2024. Observed is **−14% (−27% to +5%)** against Control Markets (excluding Russia and Belarus, under sanctions). With one provisional post year, this only confirms the direction of the EU–Japan result.

**What this means.** The monotone constraint guarantees that an agreement never lowers Export Potential (ADR 0002). It does not make the size right. Scenario Effects should be read as an upper-end association, about the order of magnitude, not a forecast. The Scenarios page now states the EU–Japan result next to every Scenario, so Advisors see this before they quote a number.

**Conclusion.** The evaluation was set up so that it could fail, and in places it did. The model clearly beats the naive baseline across all Exporters, but for UK pairs scored out-of-fold its ranking gain is narrow (0.54 against 0.52); its value there is mainly the better-calibrated *size* of Export Potential. Large Export Gaps concentrate growth (Lift 6.6), but no more than pairs of the same size do (0.83). Scenario Effects overstate the change seen after real agreements several-fold. The app states each of these findings next to the numbers they qualify (course: *no signal is a result, not a failure*), and the model card in section 10 records them.

## 8. Explaining Export Potential

An Advisor who takes an Opportunity to a firm must say *why* the model expects more trade. The course distinguishes **global** explanations (what drives the model overall) from **local** ones (why this prediction), and recommends **SHAP** for local explanations that are additive and faithful to the model being shipped. Both are computed here.

XGBoost computes exact TreeSHAP values itself (`pred_contribs=True`), so no approximation or background sample is needed. Each pair is explained by **its own fold model**, the one that held it out and scored its Baseline (ADR 0001): explaining any other model would explain a number the Advisor never sees (course: *explain the model you ship*). The model has a log link, so contributions add up on the log scale, and `exp` of the base value plus every contribution reproduces the Baseline Export Potential exactly (`tests/test_explain.py` checks this for every fixture pair; on the real artefact the error is below 3e-5 over 200 pairs). On the dollar scale each factor multiplies Export Potential by `exp(contribution)`.

**Global importance** is the mean absolute contribution over a random sample of 20,000 Baseline pairs. Unlike impurity importance, it is measured on the shipped predictions and cannot rank a feature the model never uses (course: *feature importance: impurity vs permutation*).

In [ ]:
from exportspredictor import explain

importance = explain.global_importance(bundle, sample=20_000)
importance.with_columns(typical_multiplier=pl.col("mean_abs_contribution").exp())

On the real artefact (20,000 pairs, about a minute on the CPU) demand dominates: how much the Market imports the Product from others moves a typical prediction by a factor of about 10. Then come UK supply of the Product to other Markets (×2.6), the high-income share of the Market's imports (×1.9), the Market's GDP (×1.9), distance (×1.7) and UK specialisation (×1.6). The policy features matter much less on average: a Trade Agreement moves a typical pair by ×1.18, and the CPTPP and EFTA flags by about 1%. This is expected, since most pairs' policy status never changes, but it also shows that Scenario Effects rest on a small part of the model.

Low global importance is also a governance signal (course: mean |SHAP| *supports minimisation*). The EFTA and CPTPP flags and the common-colonizer feature contribute about 1–2% to a typical prediction. They stay in the model because Scenarios need the bloc flags, but a future version could drop common colonizer at no cost.

**A local explanation.** The largest Opportunity on the real data is packaged medicines (`300490`) to China. The Why? page shows the same factors in the same words:

In [ ]:
top_opportunity = baseline.sort("export_gap_usd", descending=True).row(0, named=True)
why = explain.explain_pair(bundle, top_opportunity["hs6"], top_opportunity["iso3"])
print(f"{top_opportunity['description'][:60]} to {why.market}: Export Potential {explain.usd(why.export_potential_usd)}, scored by fold model {why.fold}")
display(pl.DataFrame([{"factor": f.label, "value": f.value_text, "multiplier": f.multiplier} for f in why.factors[:8]]))
pl.DataFrame([{"year": point.year, "actual_exports_usd": point.actual_exports_usd} for point in why.history])

On the real data the model expects **US$4.9bn** of UK packaged medicines in China against Actual Exports of $0.57bn in 2024. The explanation reads as an Advisor would want: the UK exports $13.6bn of this Product to other Markets (×40) and China imports $13.9bn of it from others (×33), both large economies (×4.1 and ×2.2), and the UK is specialised in it (1.9 times the world share, ×1.6); distance pulls it down (×0.72). The history adds what the model cannot see: UK exports to China fell from $1.17bn in 2021 to $0.57bn in 2024, so a large part of this gap is recent, and the reasons (pricing, procurement, regulation) need checking before it is pitched as an Opportunity.

Two limits apply to every explanation. SHAP values are **associations inside the model**, not causes: "distance lowers Export Potential" means comparable distant pairs trade less. And several top factors (the Market's size, its imports, distance) are not things an Advisor can change, which the course flags as a test of *actionable* explanations. The actionable reading is the supply side: the UK already sells this Product well elsewhere.

**Conclusion.** Every Export Potential can be decomposed exactly into named factors from the fold model that produced it, in words an Advisor can repeat. Globally the model is driven by what Markets buy and what the UK sells; policy features carry a small part of each prediction, which is consistent with Scenario Effects being sensitive to how the model learned them (section 7). Local explanations combined with the Actual Exports history are the Advisor's main defence against a misleading Opportunity.

## 9. Saving the artefacts

The API must serve exactly what was evaluated (course: *registry versions* and *reproducible artefacts*). The API serves exactly what was trained here: a versioned artefact folder with the five fold models, the Baseline table, the Baseline's model inputs and folds, the Actual Exports history, the Product and Market lookups, the evaluation report of section 7 (`evaluation.json`), the model card of section 10 (`model_card.md`), and a metadata file (sources, sampling rates, the search, the chosen parameters, device and timings).

In [ ]:
saved_to = artefacts.save(bundle, artefacts_dir())
saved_to

**Conclusion.** Each run writes a new folder named by its UTC timestamp, and the API loads the newest complete one (the metadata file is written last, so a half-written folder is never loaded). Rolling back means removing or renaming the newest folder. Steps that add results to a trained model (Blocked Opportunity values, the evaluation, the backtests) copy the folder to a new version rather than editing it in place, and record `copied_from`, so every version traces back to the training run that produced its models.

## 10. Governance

The course treats responsible AI as **design constraints with owners and evidence**: "a commitment without an artefact is an aspiration". This section produces those artefacts for the model: a model card generated from the evaluated artefact, data lineage, a bias review across regions and reporting quality, the regulatory classification under the EU AI Act and the UK's approach, and the drift checks and yearly refresh that keep the model honest after release.

### Model card

The course asks for a short structured card covering intended and out-of-scope use, owner, metrics by group, limitations and data sources, *generated from code and versioned with the model*, printing UNASSIGNED rather than omitting a section. `governance.model_card` builds it from the artefact's metadata and evaluation report, so its numbers are the evaluated numbers. It also cites, in substance, the decisions a reader of the model needs (the ADRs and the project's decision log). `write_model_card` saves it as `model_card.md` in the artefact folder, next to the models it describes.

In [ ]:
from IPython.display import Markdown

from exportspredictor import governance

card_path = governance.write_model_card(bundle, saved_to)
Markdown(card_path.read_text(encoding="utf-8"))

The owner is **UNASSIGNED** on purpose: this is a course project, and naming who answers for the model is a decision for the team that adopts it. The course's warning applies: governance usually fails "at the name, not the technology".

### Data lineage

Every number the API serves should trace back to exact inputs (course: *data lineage and reproducible artefacts*). The lineage has three links:

1. **Raw downloads.** `governance.data_lineage()` fingerprints every file under `raw/` with its size and SHA-256, so a changed download is detectable. The source URLs and versions are in each package module (`baci.BACI_URL`, `gravity`, `policy`, `hmrc`).
2. **Derived data.** Every derived table is rebuilt from the raw files by one `python -m exportspredictor.<module>` command, in the order of `scripts/rebuild.py`; none is edited by hand.
3. **Artefacts.** The metadata records the BACI release, the zero-sampling rate and seed, the non-UK sampling rate, the search, the chosen parameters, the XGBoost version, the device and the timings, and `copied_from` links each version to the one it came from.

In [ ]:
if FIXTURE_MODE:
    print("Fixture mode: the fixture world is generated in code (exportspredictor.fixture), so there are no raw files to fingerprint.")
else:
    lineage = governance.data_lineage()  # hashes about 2.8 GB of downloads, a few seconds
    display(lineage)
{key: bundle.metadata.get(key) for key in ("version", "source", "xgboost_version", "device", "zero_sampling_rate", "zero_sampling_seed", "non_uk_sampling_rate", "created_at", "copied_from")}

On the real data the lineage lists 18 raw files, 2.8 GB in all; the BACI archive alone is 1,267,950,839 bytes. The artefact chain runs from the training run `20261117T002435Z` through the Blocked Opportunity values, the evaluation and the backtests to the version the API serves. A gap remains: the artefact does not yet store the raw-file hashes themselves, so proving which download a version was trained on relies on the data folder not having changed since. Writing the lineage table into the metadata at training time is a small next step.

### Bias review: regions and reporting quality

Bias here concerns countries and Sectors, not people: the model holds no personal data. The course's two families still apply:

- **Data bias.** BACI is built from what countries report, and poorer countries report later, less completely and with larger errors. Absent Flows are read as zeros, so a poorly reporting Market looks as if it buys little (measurement and representation bias).
- **Algorithmic bias.** One model, one loss and one ranking serve all Markets, and the loss is dominated by large Flows, which are concentrated in high-income Markets (objective bias).

The review follows the course's **representation audit** (group sizes and denominators first, thin groups flagged) and **worst-group evaluation**, on the UK's 2023 out-of-fold grid. World Bank **income level** stands in for reporting quality, because no per-country reporting-quality score is published with BACI. A group is flagged *thin* when it has fewer than 500 UK Flows, the course's size floor.

In [ ]:
uk_accuracy = report["accuracy"]["uk_out_of_fold"]
bias_columns = ["group", "rows", "non_zero_flows", "actual_usd", "model_spearman_within_product", "naive_spearman_within_product"]
for breakdown in ("by_region", "by_income"):
    if uk_accuracy.get(breakdown):
        display(pl.DataFrame(uk_accuracy[breakdown]).select(bias_columns).with_columns(thin=pl.col("non_zero_flows") < 500))

On the real data (UK pairs, 2023, within-Product rank correlation, model against naive):

| Income level of the Market | UK pairs | UK Flows | Share of UK exports | Model | Naive |
|---|---|---|---|---|---|
| High income | 405,522 | 180,869 | 80% | **0.65** | 0.60 |
| Upper middle income | 301,542 | 81,922 | 15% | 0.43 | 0.43 |
| Lower middle income | 244,353 | 49,073 | 4% | **0.41** | 0.36 |
| Low income | 129,975 | 12,849 | 0.3% | 0.24 | 0.24 |
| Unclassified (small territories) | 88,383 | 2,961 | 0.03% | 0.21 | 0.02 |

What the review finds:

- **A steep gradient by income.** The model ranks Markets much better in high-income Markets (0.65) than in low-income ones (0.24). By region the same pattern shows: North America 0.83 and Europe 0.66, against Latin America 0.33 and Sub-Saharan Africa 0.35.
- **It is mostly a data effect, not the model treating poorer Markets worse.** Scored across *all* Exporters, the model's accuracy is the same for every income group of the Importer (0.69–0.71), and its gain over the naive baseline is largest for low-income Importers (0.71 against 0.21). The UK gradient comes from how little the UK sells there: in low-income Markets only 1 in 10 UK pairs has any trade, so most rankings are among near-zero Flows that BACI measures least well.
- **No group is thin by the course's floor**, but the low-income and unclassified groups together hold 0.3% of UK exports, so their metrics move with few Flows.
- **Who bears the errors.** An Advisor who reads the rankings at face value will see fewer, noisier Opportunities in low-income Markets. That could steer export support away from developing Markets for reasons of data quality rather than fundamentals.

**Mitigations in place:** the app's Model Evaluation page shows the region breakdown and the reporting-quality limitation, and every ranking can be filtered by region, so Advisors can compare Opportunities within a region rather than across them. A per-region or per-income threshold for Opportunities (course: *per-group thresholds*) would be the next step if the rankings were used to allocate support budgets.

### Regulation: the EU AI Act and the UK approach

**EU AI Act.** The course's rule is that *intended purpose decides the tier*, not the technique. The intended purpose here is decision support for public export promotion: ranking country–product pairs and exploring policy associations for officials, from aggregate trade statistics.

- It is **not prohibited**: no manipulation, social scoring or biometric use.
- It is **not high-risk**: it does not fall in any Annex III use (employment, credit, education, essential services or benefits for natural persons, law enforcement, migration, justice, democratic processes, biometrics), and it is not a safety component of a regulated product. Its outputs concern countries and Products, never natural persons.
- **Transparency duties** (Article 50, from August 2026) target systems that interact with people as if human, or generate synthetic content. The app does neither, though it labels every number as a model output and states its limits, which meets the spirit anyway.
- So it is **minimal risk**, with no new legal duties. The 2026 Digital Omnibus moved high-risk dates (stand-alone uses to December 2027) but does not change this classification.
- The Act reaches providers placing systems on the EU market or whose outputs are used in the EU. A UK government tool used in the UK falls outside it, but classifying it is still the right discipline for UK government work with EU partners.

**Function creep would change the tier.** Reusing the model to score individual firms for grants or support eligibility would make it a tool for decisions about specific businesses and, through their owners, possibly about people. That use must be re-classified first, and the model card lists it as out of scope.

**The UK approach.** There is no AI statute: existing regulators apply five cross-sector principles, and the course asks for an artefact per principle, not a promise.

| Principle | Evidence in this project |
|---|---|
| Safety, security and robustness | Time-split test year never seen in tuning (tested); Event Backtests; monotone constraints; leakage tests; the API reports healthy only once artefacts are loaded |
| Transparency and explainability | The Why? page (exact SHAP from the serving model); this model card; the limitations panel; associational wording everywhere |
| Fairness | The bias review above, by region and income level; the region breakdown on the Model Evaluation page |
| Accountability and governance | Versioned artefacts with lineage; decisions recorded in ADRs and a decision log; the owner is still UNASSIGNED, which is the gap to close before use |
| Contestability and redress | Outputs are leads for Advisors, who decide; every Opportunity can be challenged through its explanation and Actual Exports history |

**Data protection.** The data are country-level trade aggregates with no personal data, so UK GDPR (including Article 22 on solely automated decisions) is not engaged and no DPIA is needed. As a tool that informs public-sector decisions, it would be a candidate for a record under the UK's Algorithmic Transparency Recording Standard, for which the model card is most of the content.

### Drift and the yearly refresh

A trained model is frozen while trade moves (course: *model drift and its three types*). For this model:

- **Data drift**, P(X): the inputs shift as economies grow and agreements change. The Population Stability Index (course: *PSI*) compares each feature's distribution on the UK grid in two years, rated stable below 0.10, watch to 0.20 and act above.
- **Concept drift**, P(y | X): the same fundamentals bring different trade, for example after the 2025 US tariff rises or new sanctions, none of which the model sees. It shows as falling accuracy while inputs look normal, so it needs the yearly re-evaluation, not PSI.
- **Label drift**, P(y): BACI revises its latest year, and the provisional 2024 already has 5.5% fewer Flows than 2023.

In [ ]:
if FIXTURE_MODE:
    last_year = feature_table.select(pl.col("year").max()).collect().item()
    drift_reference = feature_table.filter(pl.col("year") == last_year - 1).collect()
    drift_current = feature_table.filter(pl.col("year") == last_year).collect()
else:
    drift_reference = pl.read_parquet(feature_dir / "uk_grid_2023.parquet")
    drift_current = pl.read_parquet(feature_dir / "uk_grid_2024.parquet")
governance.feature_drift(drift_reference, drift_current, features.FEATURES)

On the real UK grid, 2023 against 2024, every Market-side and Product-side feature is **stable** (all PSI below 0.01; the largest is the high-income share of imports at 0.007). Two features read **act**, and both are expected: the UK's own GDP and population take a single value per year on the UK grid, so any change moves every row to a new bin. PSI is the wrong test for a constant; the right check for them is that the new value lies inside the clip range fitted on training years, which `FeatureTransform` enforces. The policy features did not move between 2023 and 2024; they will in 2025, when the UK's CPTPP accession takes effect.

**The yearly refresh.** CEPII publishes a new BACI release each January, revising the provisional year and adding one more.

1. **Rebuild** from scratch with `scripts/rebuild.py` after updating `baci.BACI_VERSION` (and the HMRC year for the cross-check). The previous artefact stays in place until the new one is complete.
2. **Check inputs before training**: re-run the HMRC cross-check and the PSI table above. A large PSI on a Market-side feature is a question before it is a reason to retrain: a broken download or a changed code list must be fixed, not trained on (course: *responding to a drift alarm*).
3. **Retrain and re-evaluate** with the test year moved on by one. Promote the new version only if it still beats the naive baseline on the headline metric, and compare the new Baseline's rankings with the old one (the stability check of section 7) so that Advisors are told when the top Opportunities move.
4. **Use the new year as a live test.** The UK's CPTPP accession counts from 2025, so the first BACI release with 2025 data turns the CPTPP Scenario into a real Event Backtest of a UK agreement.
5. **Roll back** by removing the newest artefact folder; the API then serves the previous version.

Each step needs a named owner, which, like the model's owner, is still UNASSIGNED.

**Conclusion.** The model is minimal risk under the EU AI Act for its intended purpose and fits the UK's principles with evidence for each, provided its use stays decision support for officials. Its main fairness risk is a data one: UK rankings are much weaker in low-income Markets, because BACI measures small Flows there least well, and Advisors are told so. The model card carries the evaluated numbers and the decisions behind them, versioned with the model. The open governance gaps are named rather than hidden: no owner, raw-file hashes not yet stored in the artefact, and no per-region Opportunity thresholds.

## 11. Conclusions

The project built an end-to-end system: data from four primary sources rebuilt by one script, a feature table shared by training and serving, five out-of-fold gradient-boosted models trained on the GPU, a FastAPI service, and a Streamlit app for Advisors. This section sums up what the evidence supports, what goes beyond the course, and what to do next.

**What the evidence supports**

- The model learns global trade well (0.71 within-Product rank correlation across all Exporters on a year it never saw, against 0.56 for the naive baseline), and estimates the size of UK Flows much better than the baseline.
- For UK pairs the ranking gain over a simple market-share rule is narrow (0.54 against 0.52). The honest pitch to Advisors is a well-calibrated map of *where comparable exporters sell more*, with explanations, not a ranking that beats common sense by much.
- Large Export Gaps concentrate later growth (Lift 6.6), but no more than other pairs of the same size (0.83). An Opportunity marks a Market worth a look.
- Scenario Effects are associations several times larger than the changes observed after EU–Japan (+45% against +3%) and UK–Australia/New Zealand (+27% against −14%). They indicate direction and an upper-end order of magnitude.
- BACI's UK data differ from HMRC's by explained amounts, and rank pairs alike (0.89).

**Beyond the course notes:** out-of-fold cross-fitting for the target country (ADR 0001); Tweedie loss with Horvitz–Thompson weighted zero sampling; monotone-constrained boosting so that Scenarios respect the direction of policy (ADR 0002); training on data larger than memory through a GPU data iterator; Lift with a cluster bootstrap and a size-matched control; difference-in-differences Event Backtests against Control Markets; an independent primary-source cross-check of the training data; and a bias review by reporting quality.

**Next steps**, in order of value:

1. **Calibrate Scenario Effects** against a structural gravity (PPML) estimate or a pool of Event Backtests, and report a range rather than a point.
2. **Add sanctions and tariffs** as features (the research for tariffs is in `docs/research/hs6-tariff-data.md`), so that Russia-style Opportunities disappear and tariff Scenarios become possible.
3. **Train on every row** with XGBoost's external-memory mode instead of the 30% non-UK sample.
4. **Name an owner**, store the lineage hashes in the artefact, and run the yearly refresh with the 2025 data as the first live test of the CPTPP Scenario.

**Conclusion.** The UK Exports Predictor is a sound, transparent tool for finding *where to look*, built and evaluated so that its weaknesses are measured rather than hidden. Its numbers should be read as associations learned from global trade: useful for prioritising conversations with industry, not for promising what a market or an agreement will deliver.